In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # earth radius in km
    φ1, φ2 = np.radians(lat1), np.radians(lat2)
    Δφ     = np.radians(lat2 - lat1)
    Δλ     = np.radians(lon2 - lon1)
    a = np.sin(Δφ/2)**2 + np.cos(φ1)*np.cos(φ2)*np.sin(Δλ/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

In [ ]:
# pull in the json data. each json object is on a new line
df = pd.read_json('bulk_synthetic_entra_signin_last_1_year copy.ndjson', lines=True)

In [ ]:
# need to flatten the json into a table.  nested objects - column names will be name.nest.nest
df = pd.json_normalize(df.to_dict(orient="records"), sep=".")

In [ ]:
# insure @timestamp is datetime convert to datetime
df["@timestamp"] = pd.to_datetime(df["@timestamp"], utc=True, errors="coerce")

# floor the @timestamp values to day
df["login_day"] = df["@timestamp"].dt.normalize()

In [ ]:
df = df.sort_values(["user.name", "@timestamp"])

In [ ]:
# shift previous coords and timestamps to new columns
df["lat_prev"] = df["geo.location.lat"].shift()
df["lon_prev"] = df["geo.location.lon"].shift()
df["ts_prev"]  = df["@timestamp"].shift()


In [ ]:
# compute distance (km) and time difference (hours)
df["dist_km"] = haversine(df["lat_prev"], df["lon_prev"],
                          df["geo.location.lat"], df["geo.location.lon"])

df["hours"]   = (df["@timestamp"] - df["ts_prev"]).dt.total_seconds() / 3600.0


In [ ]:
# compute speed, guarding against zero or negative elapsed time
df["speed_kmh"] = df["dist_km"] / df["hours"]
df.loc[df["hours"] <= 0, "speed_kmh"] = 0.0

In [ ]:
# Group by your user_name and login_day, then pull out:
#
# max_speed_kmh
#
# last timestamp
#
# last IP
#
#last lat/lon
#
# remmeber last works because perviously sorted by @timestamp so time is chronilogical

result = (
    df
    .groupby(["user.name", "login_day"])
    .agg(
        max_speed_kmh = ("speed_kmh", "max"),
        last_ts       = ("@timestamp", "max"),
        last_ip       = ("client.ip",   "last"),
        last_lat      = ("geo.location.lat", "last"),
        last_lon      = ("geo.location.lon", "last"),
    )
    .reset_index()
)


In [ ]:
impossible_travel = result[result["max_speed_kmh"] > 0]

In [ ]:
impossible_travel.sort_values(by=['max_speed_kmh'], ascending=False)